# Fase 5 — Orquestación end-to-end

Orden: **Bronze → Silver → Gold → Calidad**.

Modos: `full` (snapshot) | `incremental` (watermark + MERGE). Ver `docs/incrementalidad.md`.

In [ ]:
%pip install openpyxl

In [ ]:
dbutils.widgets.text("repo_root", "", "Ruta Repo Databricks")
dbutils.widgets.text("raw_volume_path", "/Volumes/ips_analytics/raw/raw_data/", "Volume Excel")
dbutils.widgets.text("batch_id", "", "Batch ID (vacío = auto)")
dbutils.widgets.dropdown("load_mode", "full", ["full", "incremental"], "Modo carga")
dbutils.widgets.dropdown("run_gold", "true", ["true", "false"], "Ejecutar Gold")
dbutils.widgets.dropdown("fail_on_quality", "true", ["true", "false"], "Fallar si QC rojo")

In [ ]:
import sys
repo_root = dbutils.widgets.get("repo_root").rstrip("/")
if not repo_root:
    try:
        nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        if "/notebooks/" in nb_path:
            repo_root = nb_path.split("/notebooks/")[0]
    except Exception:
        pass
if not repo_root:
    raise ValueError("Widget repo_root vacío: usa /Workspace/Repos/<usuario>/<repo>")
sys.path.insert(0, f"{repo_root}/src")

from ips_analytics.config import generate_batch_id
from ips_analytics.pipeline.orchestrate import run_end_to_end_pipeline

batch_id = dbutils.widgets.get("batch_id").strip() or None
load_mode = dbutils.widgets.get("load_mode")
run_gold = dbutils.widgets.get("run_gold") == "true"
fail_on_qc = dbutils.widgets.get("fail_on_quality") == "true"

result = run_end_to_end_pipeline(
    spark,
    raw_volume_path=dbutils.widgets.get("raw_volume_path"),
    batch_id=batch_id,
    load_mode=load_mode,
    run_gold=run_gold,
    run_quality=True,
    fail_on_quality=fail_on_qc,
)

print(f"batch_id={result.batch_id} mode={result.load_mode}")
print(f"bronze={result.bronze_ok} silver={result.silver_ok} gold={result.gold_ok}")
if result.quality:
    for line in result.quality.summary_lines():
        print(line)
    print("QC passed:", result.quality.passed)

In [ ]:
from ips_analytics.config import DEFAULT_CONFIG
spark.table(f"{DEFAULT_CONFIG.catalog}.ops.ingestion_watermark").show(truncate=False)